# 04 · Plotting with gog4j

Numbers became a table; now the table becomes a picture. **gog4j** is ggplot2
for JavaFX. The grammar is three pieces:

* **data** — here a `HardwoodDataFrame`, read straight from the Parquet
* **aes** — which columns drive x, y, colour, …
* **geoms** — how to draw them (points, lines, bars, density, …)

Build a plot with `Ggplot.ggplot(...)`, add layers with `.geoms(...)`, then
either `display(...)` it or export it to SVG.

In [0]:
addDependency("org.jtaccuino:gog4j:0.5-SNAPSHOT");
addDependency("org.jtaccuino:gog4j-hardwood:0.5-SNAPSHOT");

// the jars above are on the class path for the snippets that follow; name any
// that did not land, rather than failing later with "cannot find symbol"
try { Class.forName("org.jtaccuino.gog.labs.Labs"); }
catch (ClassNotFoundException notThere) {
    println("warning: org.jtaccuino.gog.labs.Labs is not on the class path yet (%s)",
            notThere.getMessage());
}

import org.jtaccuino.gog.*;
import org.jtaccuino.gog.labs.Labs;
import org.jtaccuino.gog.render.SvgExporter;
import org.jtaccuino.gog.theme.Theme;
import org.jtaccuino.gog.hardwood.HardwoodDataFrame;
import java.nio.file.Files;
import java.nio.file.Path;

// The notebook may live in a sub-folder (exercises/, bonus/, ...), so walk up
// from the notebook's own folder to find the repository's data/ directory.
Path dataset() {
    for (Path p = cwd.toAbsolutePath(); p != null; p = p.getParent()) {
        Path candidate = p.resolve("data/celestrak_gp_catalog.parquet");
        if (Files.isRegularFile(candidate)) return candidate;
    }
    throw new IllegalStateException("data/celestrak_gp_catalog.parquet not found above " + cwd);
}

var full = HardwoodDataFrame.of(dataset());

// Project to just the fields this module plots. Reading and keeping only the
// columns you need is the columnar habit; here it also keeps the frame small.
var table = HardwoodDataFrame.ofColumns(java.util.Map.of(
        "inclination", full.column("inclination"),
        "eccentricity", full.column("eccentricity"),
        "period_min", full.column("period_min"),
        "apogee_km", full.column("apogee_km"),
        "orbit_class", full.column("orbit_class"),
        "satcat_object_type", full.column("satcat_object_type")));

// Everything that touches JavaFX — building the plot's canvas, exporting,
// attaching a node — must run on the FX Application Thread. Notebook code runs
// on JShell's worker thread, so marshal with onFx(...) and wait for it.
void onFx(Runnable work) {
    if (javafx.application.Platform.isFxApplicationThread()) {
        work.run();
        return;
    }
    var latch = new java.util.concurrent.CountDownLatch(1);
    javafx.application.Platform.runLater(() -> {
        try { work.run(); } catch (Throwable t) { t.printStackTrace(); } finally { latch.countDown(); }
    });
    try { latch.await(); } catch (InterruptedException e) { Thread.currentThread().interrupt(); }
}

void save(GgFigure figure, String name) {
    save(figure, 1200, 800, name);
}

void save(GgFigure figure, double width, double height, String name) {
    var path = cwd.resolve(name);
    var size = new long[]{-1};
    var error = new String[]{null};
    onFx(() -> {
        try {
            new SvgExporter().size(width, height).batchPoints(true).write(figure, path);
            size[0] = Files.size(path);
        } catch (Exception e) {
            error[0] = String.valueOf(e);
        }
    });
    if (error[0] != null) {
        println("could not save %s: %s", name, error[0]);
    } else {
        println("saved %s (%s bytes)", name, String.format(java.util.Locale.ROOT, "%,d", size[0]));
    }
}

## 1 · Data + aes + geoms

One point per satellite: inclination across, apogee up, coloured by orbit class.
That single `color(...)` is the whole point of a grammar of graphics.

In [0]:
Plot<HardwoodDataFrame> p = Ggplot.ggplot(table,
        Aes.aes().x("inclination").y("apogee_km").color("orbit_class"))
    .geoms(Geoms.point());

In [0]:
// ── given ── look at it, and save a copy
onFx(p::markDirty);
display(p);
save(p, "01-scatter.svg");

## 2 · Labels

A plot without axis labels is a riddle. `Labs.labs(...)` answers it.

In [0]:
p = p.labs(Labs.labs("Everything in orbit", "Inclination (deg)", "Apogee (km)"));

In [0]:
onFx(p::markDirty);
display(p);
save(p, "02-labelled.svg");

## 3 · Theme

A theme changes every colour at once. On a projector, dark wins.

In [0]:
p = p.theme(Theme.theme_dark());

In [0]:
onFx(p::markDirty);
display(p);
save(p, "03-dark.svg");

## 4 · Facets

Faceting splits one plot into a small multiple per category — the fastest way to
see that the classes are genuinely different populations.

In [0]:
p = p.facets(Facets.wrap("orbit_class", 2));

In [0]:
onFx(p::markDirty);
display(p);
save(p, "04-facets.svg");

## 5 · Reference lines

The geostationary belt sits at 35 786 km. `Geoms.hline(...)` draws it, and one
glance then explains the horizontal stripe of GEO objects.

In [0]:
p = p.geoms(Geoms.point(), Geoms.hline(35786));

In [0]:
onFx(p::markDirty);
display(p);
save(p, "05-geobelt.svg");

## 6 · Export

`display` is for you; `SvgExporter` is for the paper. Export the finished figure
at print size, with the 21 000 points batched so the file stays editable.

In [0]:
save(p, 1600, 1000, "celestrak-orbits.svg");

## The bridge to machine learning

Everything so far *described* the catalog. Now predict something from it. Can the
orbital elements alone tell debris from a working satellite?

`satcat_object_type` gives the ground truth: `PAY` (payload) or `DEB` / `R/B`
(debris). The features are the elements — inclination, eccentricity, period, mean
motion, drag. We stop at the train/test split — a labelled feature matrix, ready
for a model.

In [0]:
// ── given ── turn it into a supervised problem
addDependency("org.dflib:dflib:2.0.0-M7");
addDependency("org.dflib:dflib-parquet:2.0.0-M7");

// the jars above are on the class path for the snippets that follow; name any
// that did not land, rather than failing later with "cannot find symbol"
try { Class.forName("org.dflib.DataFrame"); }
catch (ClassNotFoundException notThere) {
    println("warning: org.dflib.DataFrame is not on the class path yet (%s)",
            notThere.getMessage());
}

import org.dflib.DataFrame;
import org.dflib.parquet.Parquet;
import static org.dflib.Exp.*;

var df = Parquet.loader().load(dataset());

// keep only rows with a usable label
var labelled = df.rows(
        $str("satcat_object_type").eq("PAY")
            .or($str("satcat_object_type").eq("DEB"))
            .or($str("satcat_object_type").eq("R/B")))
    .select();

println("labelled %s of %s rows", String.format(java.util.Locale.ROOT, "%,d", labelled.height()),
        String.format(java.util.Locale.ROOT, "%,d", df.height()));
System.out.println(labelled.group("satcat_object_type").agg($col("satcat_object_type"), count()));

In [0]:
// ── given ── features and a deterministic split
var features = labelled
    .cols("inclination", "eccentricity", "period_min", "mean_motion", "bstar")
    .select();

int split = (int) (features.height() * 0.8);
var train = features.rowsRange(0, split).select();
var test  = features.rowsRange(split, features.height()).select();

println("train %s   test %s   features %d",
        String.format(java.util.Locale.ROOT, "%,d", train.height()),
        String.format(java.util.Locale.ROOT, "%,d", test.height()),
        features.width());
println("→ a feature matrix and a labelled split, ready for a model.");

## ★ BONUS 1 — Pairs and composition

Two dozen features at once, as a scatterplot matrix. `matrixPlot` does the grid;
`composedPlot` arranges finished figures side by side.

In [0]:
// ── ★ BONUS 1 ─────────────────────────────────────────────────────────
var pairs = Ggplot.matrixPlot(table,
    Aes.aes().color("orbit_class"),
    "inclination", "eccentricity", "period_min", "apogee_km");
display(pairs);
save(pairs, "bonus-pairs.svg");

## ★ BONUS 2 — A three-dimensional orbit cloud

Add inclination, period and eccentricity as x/y/z and let the points fill the
volume.

In [0]:
// ── ★ BONUS 2 ─────────────────────────────────────────────────────────
var cloud = Ggplot.ggplot3d(table,
        Aes.aes().x("inclination").y("period_min").z("eccentricity").color("orbit_class"))
    .geoms(Geoms.point3d());
display(cloud);
save(cloud, "bonus-3d.svg");

## ★ BONUS 3 — The wall chart

A filled 2-D density instead of 21 000 points, with the GEO belt drawn on top.
This is the figure that goes on the poster.

In [0]:
// ── ★ BONUS 3 ─────────────────────────────────────────────────────────
var wall = Ggplot.ggplot(table,
        Aes.aes().x("inclination").y("apogee_km").fill("orbit_class"))
    .geoms(Geoms.density2dFilled(), Geoms.hline(35786))
    .labs(Labs.labs("Where satellites live", "Inclination (deg)", "Apogee (km)"))
    .theme(Theme.theme_dark());
display(wall);
save(wall, "bonus-wallchart.svg");

That is the whole pipeline: **Hardwood** read it, **dflib** shaped it,
**gog4j** drew it, and you left a labelled train/test split on the table for the
model.

Back to the [overview](../narrative/00-overview.md).